# RNN MFCC-10
Trains multilabel and binary UUV BiLSTMs with grouped 5-fold cross-validation on MFCC-10 data.

In [ ]:
import sys
from pathlib import Path

UTILS_GITHUB_RAW_BASE_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/utils"
COMMON_UTILS_FILE = "common_utils.py"
MODEL_UTILS_FILE = "rnn_utils.py"
MODEL_DIR = "RNN"
common_dirs = [Path.cwd() / "utils", Path.cwd().parent / "utils", Path("/content/utils"), Path("/content/drive/MyDrive/STUDA/src/utils")]
model_dirs = [Path.cwd(), Path.cwd() / MODEL_DIR, Path.cwd().parent / MODEL_DIR, Path("/content") / MODEL_DIR, Path("/content/drive/MyDrive/STUDA/src") / MODEL_DIR]
common_dir = next((directory for directory in common_dirs if (directory / COMMON_UTILS_FILE).exists()), None)
model_dir = next((directory for directory in model_dirs if (directory / MODEL_UTILS_FILE).exists()), None)

if (common_dir is None or model_dir is None) and UTILS_GITHUB_RAW_BASE_URL:
    import urllib.request
    common_dir, model_dir = Path("/content/utils"), Path("/content") / MODEL_DIR
    common_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{UTILS_GITHUB_RAW_BASE_URL}/common_utils.py", common_dir / COMMON_UTILS_FILE)
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/{MODEL_DIR}/{MODEL_UTILS_FILE}", model_dir / MODEL_UTILS_FILE)

if common_dir is None or model_dir is None:
    raise FileNotFoundError("Required common and model utility files were not found.")

sys.path.insert(0, str(common_dir))
sys.path.insert(0, str(model_dir))


In [ ]:
import pandas as pd
import tensorflow as tf
from IPython.display import display
from google.colab import files

from common_utils import (
    configure_kaggle_access, cross_validate_keras_models_for_variants,
    evaluate_models_for_variants, extract_zip, plot_training_histories,
    prepare_mfcc_dataset_variants, save_keras_artifacts,
    summarize_cross_validation, train_final_keras_models_for_variants,
    zip_artifacts,
)
from rnn_utils import build_mfcc_rnn, get_rnn_callbacks


In [ ]:
import os
import tensorflow as tf

print("COLAB_TPU_ADDR:", os.environ.get("COLAB_TPU_ADDR"))
print("TPU devices:", tf.config.list_logical_devices("TPU"))

In [ ]:
DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")
EPOCHS = 50
BATCH_SIZE = 64


In [ ]:
configure_kaggle_access("Kaggle")
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
extract_zip(ARCHIVE_PATH, "/content")
DATA_PATH = Path("/content") / f"{DATASET_KEY}.npz"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected feature archive: {DATA_PATH}")
print(f"Using feature archive: {DATA_PATH}")


In [ ]:
cv_dataset = prepare_mfcc_dataset_variants(DATA_PATH)
final_variants = cv_dataset.final_variants()
print(f"Split ID: {cv_dataset.split_id}")
print(f"CV folds: {cv_dataset.n_splits}")
print(f"MFCC input shape: {cv_dataset.normal.cv_data.shape[1:]}")


In [ ]:
cv_multilabel_results, multilabel_best_epochs, multilabel_cv_histories = cross_validate_keras_models_for_variants(
    build_mfcc_rnn, cv_dataset, "multilabel", EPOCHS, BATCH_SIZE,
    get_rnn_callbacks, DATASET_LABEL,
)
multilabel_models, multilabel_histories = train_final_keras_models_for_variants(
    build_mfcc_rnn, cv_dataset, "multilabel", multilabel_best_epochs, BATCH_SIZE,
)
display(summarize_cross_validation(cv_multilabel_results))


In [ ]:
cv_binary_results, binary_best_epochs, binary_cv_histories = cross_validate_keras_models_for_variants(
    build_mfcc_rnn, cv_dataset, "binary", EPOCHS, BATCH_SIZE,
    get_rnn_callbacks, DATASET_LABEL,
)
binary_models, binary_histories = train_final_keras_models_for_variants(
    build_mfcc_rnn, cv_dataset, "binary", binary_best_epochs, BATCH_SIZE,
)
display(summarize_cross_validation(cv_binary_results))


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, final_variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, final_variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])


In [ ]:
plot_training_histories(multilabel_histories, f"Multilabel RNN final training - {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary RNN final training - {DATASET_LABEL}")

split_metadata = {
    "split_id": cv_dataset.split_id,
    "n_splits": cv_dataset.n_splits,
    "n_mfcc": cv_dataset.n_mfcc,
}
save_dir = save_keras_artifacts(
    f"/content/saved_artifacts/rnn_{DATASET_KEY}",
    DATASET_KEY,
    multilabel_models,
    binary_models,
    multilabel_histories,
    binary_histories,
    multilabel_results,
    binary_results,
    cv_multilabel_results=cv_multilabel_results,
    cv_binary_results=cv_binary_results,
    cv_histories={**multilabel_cv_histories, **binary_cv_histories},
    split_metadata=split_metadata,
)
comparison_results.to_csv(save_dir / f"rnn_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/rnn_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
